# AliExpress Network Research Platform
## Detection of Dangerous Product Listings & Review Manipulation Under the EU Digital Services Act (DSA)

This research notebook demonstrates the end-to-end Big Data and Graph Analytics methodology for:
1. Ingesting e-commerce marketplace graphs & EU Safety Gate (RAPEX) regulatory ground truth.
2. Constructing Bipartite $(U_{\text{reviewers}}, V_{\text{products}}, E)$ and Co-Review Projected Networks.
3. Running **Personalized PageRank Graph Risk Diffusion** to calculate the **Product Hazard Index (PHI)**.
4. Performing **Louvain Modularity Community Detection** to uncover rogue merchant collusion rings.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import networkx as nx
import plotly.graph_objects as go

from pipeline.dataset_loader import DatasetLoader
from pipeline.regulatory_safety_loader import RegulatorySafetyLoader
from pipeline.graph_builder.bipartite_graph import BipartiteGraphBuilder
from graph.communities.community_detection import CommunityDetector
from graph.risk_propagation import GraphRiskPropagator
from graph.anomalies.anomaly_detector import AnomalyDetector
from experiments.visualize_network import NetworkVisualizer

print("Modules imported successfully.")

### Step 1: Ingest Marketplace Dataset & Annotate EU Regulatory Violations

In [ ]:
loader = DatasetLoader()
df_products, df_sellers, df_reviews = loader.generate_benchmark_dataset(
    num_products=500,
    num_sellers=50,
    num_reviewers=1500,
    num_reviews=6000,
    anomaly_ratio=0.15,
    random_seed=42,
)

safety_loader = RegulatorySafetyLoader(seed=42)
df_products, df_sellers = safety_loader.annotate_products_with_regulatory_risk(df_products, df_sellers)

print(f"Loaded {len(df_products)} products, {len(df_sellers)} sellers, {len(df_reviews)} reviews.")
df_products.head(3)

### Step 2: Build Bipartite Review Network & Augment with Seller Co-Listings

In [ ]:
builder = BipartiteGraphBuilder()
B = builder.build_bipartite_graph(df_reviews, df_products)
P = builder.project_product_network(B)

propagator = GraphRiskPropagator()
G_risk = propagator.build_augmented_risk_graph(P, df_products)
df_scored = propagator.compute_product_hazard_index(G_risk, df_products)

print("Risk Diffusion Complete. Top Flagged Hazardous Products:")
df_scored[["product_id", "seller_id", "category", "hazard_code", "product_hazard_index", "risk_tier"]].head(5)

### Step 3: Louvain Modularity Community Detection & Anomaly Scoring

In [ ]:
detector = CommunityDetector()
node_to_comm, df_communities = detector.detect_communities(P)

anomaly_detector = AnomalyDetector(contamination=0.15)
df_features = anomaly_detector.extract_graph_features(P, node_to_comm, df_communities)

cols_merge = [c for c in ["product_id", "price", "hazard_code", "is_known_eu_sanction", "is_true_dangerous", "keyword_risk_flag", "product_hazard_index", "risk_tier"] if c in df_scored.columns]
df_full = df_features.merge(df_scored[cols_merge], on="product_id", how="left")
df_full = anomaly_detector.detect_anomalies(df_full)

print(f"Detected {len(df_communities)} communities. Top dense seller rings:")
df_communities.head(5)

### Step 4: Render Interactive Research Visualizations

In [ ]:
visualizer = NetworkVisualizer()
p1 = visualizer.plot_interactive_network_graph(G_risk, df_full)
p2 = visualizer.plot_evaluation_curves(df_full)
p3 = visualizer.plot_risk_distribution_dashboard(df_full)

print("Interactive dashboards exported to storage/visualizations/")